# Hybrid Quantum Model for Cross-Generator Biosignatures

This notebook adapts the original model sketch to the `crossgen_biosignatures_20260311` dataset.

- trains on TauREx `train`
- picks checkpoints on an inner split carved from TauREx `train`
- reports TauREx `val` as the in-domain test set
- reports POSEIDON as a separate cross-generator holdout evaluation
- predicts the five atmospheric abundance targets `log10_vmr_*`
- keeps a quantum bottleneck, now using a lighter 12-qubit PennyLane circuit so the remote RTX 4090 can train it end-to-end

For live remote training with batch-by-batch logs, use `models/run_crossgen_training.py` instead of executing the notebook headlessly.

## 1. Configuration

Environment-variable overrides supported by this notebook:

- `CROSSGEN_DATA_ROOT`
- `OUTPUT_DIR`
- `SEED`
- `TRAIN_BATCH_SIZE`
- `EVAL_BATCH_SIZE`
- `MAX_EPOCHS`
- `QNN_QUBITS`
- `QNN_DEPTH`
- `QUANTUM_DEVICE`
- `TRAIN_POOL_LIMIT`
- `TAU_TEST_LIMIT`
- `POSEIDON_LIMIT`
- `LOG_EVERY_BATCHES`

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.name == "models":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.crossgen_hybrid_training import (
    SAFE_AUX_FEATURE_COLS,
    TARGET_COLS,
    TrainingConfig,
    default_crossgen_data_root,
    default_output_dir,
    default_quantum_device,
    load_crossgen_dataset,
    run_training_experiment,
    split_summary,
)


def env_int(name, default):
    value = os.environ.get(name)
    return default if value is None or value == "" else int(value)


def env_optional_int(name):
    value = os.environ.get(name)
    return None if value is None or value == "" else int(value)


default_data_root = default_crossgen_data_root(PROJECT_ROOT)
default_output = default_output_dir(PROJECT_ROOT)
default_train_batch = 1024 if torch.cuda.is_available() else 64
default_eval_batch = 8192 if torch.cuda.is_available() else 256

config = TrainingConfig(
    project_root=str(PROJECT_ROOT),
    data_root=os.environ.get("CROSSGEN_DATA_ROOT", str(default_data_root)),
    output_dir=os.environ.get("OUTPUT_DIR", str(default_output)),
    seed=env_int("SEED", 42),
    train_batch_size=env_int("TRAIN_BATCH_SIZE", default_train_batch),
    eval_batch_size=env_int("EVAL_BATCH_SIZE", default_eval_batch),
    max_epochs=env_int("MAX_EPOCHS", 30),
    qnn_qubits=env_int("QNN_QUBITS", 12),
    qnn_depth=env_int("QNN_DEPTH", 2),
    quantum_device=os.environ.get("QUANTUM_DEVICE", default_quantum_device()),
    train_pool_limit=env_optional_int("TRAIN_POOL_LIMIT"),
    tau_test_limit=env_optional_int("TAU_TEST_LIMIT"),
    poseidon_limit=env_optional_int("POSEIDON_LIMIT"),
    log_every_batches=env_int("LOG_EVERY_BATCHES", 1),
)

print(json.dumps(config.to_json_dict(), indent=2))
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")

## 2. Dataset preflight

This verifies the dataset layout, split counts, and the exact safe input columns before training.

In [ ]:
labels_df, noisy_spectra, sigma_ppm, wavelength_um = load_crossgen_dataset(config.resolved_data_root())
preflight = split_summary(labels_df, wavelength_um)
print(f"Spectra tensor shape before channel expansion: {tuple(noisy_spectra.shape)}")
print(f"Sigma shape: {tuple(sigma_ppm.shape)}")
print("Safe auxiliary inputs:", SAFE_AUX_FEATURE_COLS)
print("Excluded leaky columns: trace_vmr_total, vmr_h2, vmr_he, transit_depth_noiseless")
pd.DataFrame({"field": list(preflight.keys()), "value": list(preflight.values())})

## 3. Train the hybrid model

The training path keeps a quantum bottleneck but uses a lighter 12-qubit circuit than the original 16-qubit sketch so the full TauREx split remains tractable on the remote GPU.

In [ ]:
result = run_training_experiment(config)
pd.DataFrame([result["summary"]])

## 4. Evaluation summaries

In [ ]:
result["tau_test_metrics_frame"].sort_values("target").reset_index(drop=True)

In [ ]:
result["poseidon_metrics_frame"].sort_values("target").reset_index(drop=True)

In [ ]:
result["history_frame"].tail(10)

## 5. Saved artifacts

In [ ]:
artifacts_df = pd.DataFrame({"artifact": list(result["artifacts"].keys()), "path": list(result["artifacts"].values())})
artifacts_df

In [ ]:
display(Image(filename=result["artifacts"]["loss_curve_png"]))
display(Image(filename=result["artifacts"]["rmse_curve_png"]))